In [0]:
# Cell: Download and load taxi zone lookup
import pandas as pd

# Download taxi zone lookup
zone_lookup_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Read with pandas then convert to Spark
zones_pdf = pd.read_csv(zone_lookup_url)
zones_df = spark.createDataFrame(zones_pdf)

# Save as dimension table
zones_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("jeit_kg_dev.nyc_tlc.dim_zones")

print("✅ dim_zones created!")
display(zones_df)

✅ dim_zones created!


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
# Cell: Create dim_rate_codes
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

rate_codes_data = [
    (1, "Standard rate", "Metered rate"),
    (2, "JFK", "Flat rate to/from JFK Airport"),
    (3, "Newark", "Flat rate to/from Newark Airport"),
    (4, "Nassau or Westchester", "Out of city rate"),
    (5, "Negotiated fare", "Pre-arranged price"),
    (6, "Group ride", "Shared ride discount"),
    (99, "Unknown", "Unknown or null rate code")
]

rate_codes_schema = StructType([
    StructField("rate_code_id", IntegerType(), False),
    StructField("rate_code_name", StringType(), False),
    StructField("rate_code_description", StringType(), True)
])

dim_rate_codes = spark.createDataFrame(rate_codes_data, rate_codes_schema)

dim_rate_codes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("jeit_kg_dev.nyc_tlc.dim_rate_codes")

print("✅ dim_rate_codes created!")
display(dim_rate_codes)

✅ dim_rate_codes created!


rate_code_id,rate_code_name,rate_code_description
1,Standard rate,Metered rate
2,JFK,Flat rate to/from JFK Airport
3,Newark,Flat rate to/from Newark Airport
4,Nassau or Westchester,Out of city rate
5,Negotiated fare,Pre-arranged price
6,Group ride,Shared ride discount
99,Unknown,Unknown or null rate code


In [0]:
# Cell: Create dim_payment_types
payment_types_data = [
    (0, "Flex Fare", "Flexible fare trip"),
    (1, "Credit card", "Credit/debit card payment"),
    (2, "Cash", "Cash payment"),
    (3, "No charge", "No charge trip"),
    (4, "Dispute", "Disputed fare"),
    (5, "Unknown", "Unknown payment method"),
    (6, "Voided trip", "Cancelled/voided trip")
]

payment_schema = StructType([
    StructField("payment_type_id", IntegerType(), False),
    StructField("payment_type_name", StringType(), False),
    StructField("payment_type_description", StringType(), True)
])

dim_payment_types = spark.createDataFrame(payment_types_data, payment_schema)

dim_payment_types.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("jeit_kg_dev.nyc_tlc.dim_payment_types")

print("✅ dim_payment_types created!")
display(dim_payment_types)

✅ dim_payment_types created!


payment_type_id,payment_type_name,payment_type_description
0,Flex Fare,Flexible fare trip
1,Credit card,Credit/debit card payment
2,Cash,Cash payment
3,No charge,No charge trip
4,Dispute,Disputed fare
5,Unknown,Unknown payment method
6,Voided trip,Cancelled/voided trip


In [0]:
# Cell: Create dim_time
from pyspark.sql.functions import col, when

# Create time dimension with all hours and classifications
time_data = [(h,) for h in range(24)]

dim_time = spark.createDataFrame(time_data, ["hour"])

dim_time = dim_time.withColumn("time_of_day",
    when((col("hour") >= 6) & (col("hour") < 12), "Morning")
    .when((col("hour") >= 12) & (col("hour") < 17), "Afternoon")
    .when((col("hour") >= 17) & (col("hour") < 21), "Evening")
    .otherwise("Night")
)

dim_time = dim_time.withColumn("is_rush_hour",
    when(
        ((col("hour") >= 7) & (col("hour") <= 9)) |   # Morning rush
        ((col("hour") >= 17) & (col("hour") <= 19)),  # Evening rush
        True
    ).otherwise(False)
)

dim_time.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("jeit_kg_dev.nyc_tlc.dim_time")

print("✅ dim_time created!")
display(dim_time)

✅ dim_time created!


hour,time_of_day,is_rush_hour
0,Night,false
1,Night,false
2,Night,false
3,Night,false
4,Night,false
5,Night,false
6,Morning,false
7,Morning,true
8,Morning,true
9,Morning,true


In [0]:
# Cell: Create fact_taxi_trips - Month by Month for 8GB RAM
from pyspark.sql.functions import lit, col

print("="*70)
print("     CREATING FACT_TAXI_TRIPS (8GB RAM Optimized)")
print("="*70)

# Drop table if exists
spark.sql("DROP TABLE IF EXISTS jeit_kg_dev.nyc_tlc.fact_taxi_trips")

# Only set configs we CAN change (no executor/driver memory)
spark.conf.set("spark.sql.shuffle.partitions", "50")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.files.maxPartitionBytes", "64MB")  # Smaller chunks

# Process month by month for both taxi types
years = [2023, 2024, 2025]
months = list(range(1, 13))  # 1-12
taxi_types = ["green", "yellow"]

total_chunks = len(years) * len(months) * len(taxi_types)
current_chunk = 0
first_chunk = True

for taxi_type in taxi_types:
    for year in years:
        for month in months:
            current_chunk += 1
            print(f"\n[{current_chunk}/{total_chunks}] Processing {taxi_type.upper()} {year}-{month:02d}")
            
            if taxi_type == "green":
                query = f"""
                    SELECT
                        'green' as taxi_type,
                        lpep_pickup_datetime as pickup_datetime,
                        lpep_dropoff_datetime as dropoff_datetime,
                        PULocationID as pickup_zone_id,
                        DOLocationID as dropoff_zone_id,
                        pickup_hour,
                        pickup_day_of_week,
                        day_of_week_desc,
                        time_of_day,
                        pickup_year,
                        pickup_month,
                        passenger_count,
                        trip_distance,
                        trip_duration_minutes,
                        trip_efficiency,
                        fare_amount,
                        extra,
                        mta_tax,
                        tip_amount,
                        tolls_amount,
                        improvement_surcharge,
                        total_amount,
                        fare_per_mile,
                        CAST(payment_type as int) as payment_type_id,
                        payment_type_desc,
                        CAST(RatecodeID as int) as rate_code_id,
                        rate_code_desc,
                        CAST(trip_type as int) as trip_type,
                        trip_type_desc,
                        congestion_surcharge
                    FROM jeit_kg_dev.nyc_tlc.green_taxi_silver
                    WHERE pickup_year = {year} AND pickup_month = {month}
                """
            else:  # yellow
                query = f"""
                    SELECT
                        'yellow' as taxi_type,
                        tpep_pickup_datetime as pickup_datetime,
                        tpep_dropoff_datetime as dropoff_datetime,
                        PULocationID as pickup_zone_id,
                        DOLocationID as dropoff_zone_id,
                        pickup_hour,
                        pickup_day_of_week,
                        day_of_week_desc,
                        time_of_day,
                        pickup_year,
                        pickup_month,
                        passenger_count,
                        trip_distance,
                        trip_duration_minutes,
                        trip_efficiency,
                        fare_amount,
                        extra,
                        mta_tax,
                        tip_amount,
                        tolls_amount,
                        improvement_surcharge,
                        total_amount,
                        fare_per_mile,
                        CAST(payment_type as int) as payment_type_id,
                        payment_type_desc,
                        CAST(RatecodeID as int) as rate_code_id,
                        rate_code_desc,
                        NULL as trip_type,
                        NULL as trip_type_desc,
                        congestion_surcharge
                    FROM jeit_kg_dev.nyc_tlc.yellow_taxi_silver
                    WHERE pickup_year = {year} AND pickup_month = {month}
                """
            
            try:
                if first_chunk:
                    # Create table with first chunk
                    spark.sql(f"""
                        CREATE TABLE jeit_kg_dev.nyc_tlc.fact_taxi_trips
                        USING DELTA
                        PARTITIONED BY (taxi_type, pickup_year, pickup_month)
                        AS {query}
                    """)
                    first_chunk = False
                    print(f"  ✅ Table created")
                else:
                    # Append subsequent chunks
                    spark.sql(f"""
                        INSERT INTO jeit_kg_dev.nyc_tlc.fact_taxi_trips
                        {query}
                    """)
                    print(f"  ✅ Appended")
                    
            except Exception as e:
                error_msg = str(e).lower()
                if "no data" in error_msg or "empty" in error_msg or "does not exist" in error_msg:
                    print(f"  ⚠️  No data for this month (skipped)")
                else:
                    print(f"  ❌ Error: {str(e)[:150]}")
            
            # Progress indicator every 10 chunks
            if current_chunk % 10 == 0:
                pct = current_chunk/total_chunks*100
                print(f"\n  📊 Progress: {current_chunk}/{total_chunks} chunks ({pct:.1f}%)")

print("\n" + "="*70)
print("     FACT_TAXI_TRIPS CREATION COMPLETE")
print("="*70)

# Final statistics
print("\n📊 Getting final statistics...")
try:
    total = spark.sql("SELECT COUNT(*) FROM jeit_kg_dev.nyc_tlc.fact_taxi_trips").collect()[0][0]
    print(f"Total records: {total:,}")
    
    print("\nBreakdown by taxi type:")
    spark.sql("""
        SELECT taxi_type, COUNT(*) as count
        FROM jeit_kg_dev.nyc_tlc.fact_taxi_trips
        GROUP BY taxi_type
    """).show()
    
    print("\nBreakdown by year:")
    spark.sql("""
        SELECT taxi_type, pickup_year, COUNT(*) as count
        FROM jeit_kg_dev.nyc_tlc.fact_taxi_trips
        GROUP BY taxi_type, pickup_year
        ORDER BY taxi_type, pickup_year
    """).show()
    
except Exception as e:
    print(f"Could not get statistics: {e}")

     CREATING FACT_TAXI_TRIPS (8GB RAM Optimized)

[1/72] Processing GREEN 2023-01
  ✅ Table created

[2/72] Processing GREEN 2023-02
  ✅ Appended

[3/72] Processing GREEN 2023-03
  ✅ Appended

[4/72] Processing GREEN 2023-04
  ✅ Appended

[5/72] Processing GREEN 2023-05
  ✅ Appended

[6/72] Processing GREEN 2023-06
  ✅ Appended

[7/72] Processing GREEN 2023-07
  ✅ Appended

[8/72] Processing GREEN 2023-08
  ✅ Appended

[9/72] Processing GREEN 2023-09
  ✅ Appended

[10/72] Processing GREEN 2023-10
  ✅ Appended

  📊 Progress: 10/72 chunks (13.9%)

[11/72] Processing GREEN 2023-11
  ✅ Appended

[12/72] Processing GREEN 2023-12
  ✅ Appended

[13/72] Processing GREEN 2024-01
  ✅ Appended

[14/72] Processing GREEN 2024-02
  ✅ Appended

[15/72] Processing GREEN 2024-03
  ✅ Appended

[16/72] Processing GREEN 2024-04
  ✅ Appended

[17/72] Processing GREEN 2024-05
  ✅ Appended

[18/72] Processing GREEN 2024-06
  ✅ Appended

[19/72] Processing GREEN 2024-07
  ✅ Appended

[20/72] Processing GREEN

     CREATING GOLD LAYER AGGREGATIONS

1/5 Creating agg_hourly_metrics...


com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can